# 11 — CNN rung 4, experiment 3: augmentation

**Decision this feeds** (`RESOURCES.md`): the literature is split on
augmentation for this modality — Chegodaev et al. 2026 and Boulkrinat et
al. 2025 both use random flip/rotation/brightness and report it helping
their (different) protocols; this project's own EDA (section 6b)
independently cleared L-R flip as statistically safe (doesn't corrupt
the asymmetry signal), but nothing here has actually trained with any
augmentation yet — rungs 0-3 used none. `src/augment.py`
(`random_flip`, `random_rotation`, `random_brightness_jitter`,
`augment_volume`, TDD'd 2026-09-09) composes flip + ±10° rotation
(restricted to the A-P/S-I plane so it never mixes the L-R axis — see
its docstring) + a narrow brightness jitter (data is already z-scored,
so this project uses a much narrower range than the literature's
raw-intensity values).

Wired via `dataset.DatParkinsonDataset`'s new `transform` argument, on
the **inner-train split only** — never on inner-val or the outer test
rows, matching experiment 1's oversampling rule (augmentation changes
what the model is trained on, not what it's scored against).

**Gate**: same as experiments 1-2 — nested-CV + paired bootstrap against
the **current validated CNN** (rung 3, `README.md` 2026-09-09:
mean=0.4520, sd=0.0109). No hyperparameter search (the literature's
values, already the defaults in `augment.py`) — fold-0 sanity check,
then the full 5×5 gate.

**Data handling**: this notebook loads real `.nii.gz` volumes and
row-level labels throughout, so per the AI-assistant data rule
(`README.md`) it is **[RUN ME]** — run it yourself, share back only the
printed aggregate numbers, never any per-row output.

In [1]:
# [RUN ME] -- loads real pixel data + row-level labels. Reuses the
# shared on-disk volume cache (this experiment doesn't touch
# preprocessing, so cell 08's "reused" result should hold).
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import augment
import cache
import config
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
print(f"cache {'reused' if volume_cache.was_reused else 'rebuilt'} in "
      f"{time.time() - cache_start:.1f}s for {len(uids)} volumes")

cache reused in 0.2s for 1362 volumes


In [2]:
# [RUN ME] (no data access itself). Same helper as notebooks 06/07/09/10,
# with optional augmentation on the inner-train split only via
# augment.augment_volume as DatParkinsonDataset's transform. One
# np.random.Generator per call, seeded off `seed` -- augmentation draws
# vary across epochs (the loader visits rows in different orders) but
# stay reproducible given the same seed.
def train_and_score_nested(train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            use_augmentation=False,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs])

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    transform = None
    if use_augmentation:
        aug_rng = np.random.default_rng(seed)
        transform = lambda arr: augment.augment_volume(arr, aug_rng)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels,
                                                  load_fn=volume_cache.get, transform=transform)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state

In [ ]:
# [RUN ME] -- fold-0 sanity check (no hyperparameter search -- augment.py's
# literature-grounded defaults). Confirms augmentation runs without
# error and reports a first, cheap outer-fold number before committing
# to the full 5x5 gate below.
#
# AUGMENTED_PATIENCE raised from config.PATIENCE=10 to 20: augmentation
# makes the training objective harder, and rung-3's un-augmented patience
# budget lets early stopping fire before an augmented run has converged
# (Opus review, 2026-09-10 -- fold 0 stopped at 16 epochs here, i.e. best
# epoch ~6; see project memory project_dat_parkinson_rung4_gate_review.md).
# EPOCHS stays at config.EPOCHS=50, the actual cap.
batch_size, lr = 32, 2e-3  # rung 2/3's validated winner, held fixed here
AUGMENTED_PATIENCE = 20

outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                   n_splits=config.N_FOLDS, random_state=config.SEED)
fold0_train_idx, fold0_test_idx = outer_folds[0]
fold0_train_uids = [uids[i] for i in fold0_train_idx]
fold0_train_labels = [labels[i] for i in fold0_train_idx]
fold0_train_family = [families[i] for i in fold0_train_idx]
fold0_test_uids = [uids[i] for i in fold0_test_idx]
fold0_test_labels = np.array([labels[i] for i in fold0_test_idx])

start = time.time()
probs, history, best_state = train_and_score_nested(
    fold0_train_uids, fold0_train_labels, fold0_train_family,
    fold0_test_uids, batch_size=batch_size, lr=lr, seed=config.SEED,
    use_augmentation=True, patience=AUGMENTED_PATIENCE,
)
elapsed = time.time() - start
score = evaluate.log_loss_score(fold0_test_labels, probs)
print(f"fold 0 sanity check: {len(history['val_loss'])} epochs, inner-val best="
      f"{min(history['val_loss']):.4f}, outer log loss={score:.4f}, {elapsed:.1f}s "
      f"({elapsed / len(history['val_loss']):.2f}s/epoch)")
print("rung 3's fold-0/seed-42 log loss for comparison: 0.4005 (README.md)")

In [ ]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x, with augmentation on
# the inner-train split. Same protocol as rung 3
# (notebooks/07_cnn_rung3.ipynb), except patience=AUGMENTED_PATIENCE
# (see cell above) instead of config.PATIENCE.
N_REPEATS = 5
oof_repeats_aug = []

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_train_family = [families[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        probs, history, best_state = train_and_score_nested(
            fold_train_uids, fold_train_labels, fold_train_family,
            fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
            use_augmentation=True, patience=AUGMENTED_PATIENCE,
        )
        oof_probs[test_idx] = probs
        torch.save(best_state, config.CHECKPOINT_DIR / f"rung4_augment_seed{repeat_seed}_fold{fold_i}.pt")
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)
        print(f"  seed={repeat_seed} fold={fold_i}: {len(history['val_loss'])} epochs, "
              f"outer fold log loss={fold_score:.4f}")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats_aug.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
    np.save(config.DATA_PROCESSED / f"rung4_augment_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores_aug = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_aug])
print(f"\n{N_REPEATS}-repeat augmented CNN pooled log loss: "
      f"mean={repeat_scores_aug.mean():.4f}, sd={repeat_scores_aug.std(ddof=1):.4f}")
print("current validated CNN (rung 3, README.md 2026-09-09): mean=0.4520, sd=0.0109")

In [ ]:
# [RUN ME] (no data access itself). Gate: paired per-repeat delta vs.
# the current validated CNN, using evaluate.paired_repeat_gate (fixed
# 2026-09-10 per Opus review -- the old gate here used
# paired_bootstrap_ci on a single arbitrarily-chosen repeat plus a noise
# threshold scaled by a single repeat's sd instead of the mean's
# standard error; see project memory
# project_dat_parkinson_rung4_gate_review.md).
y_true = np.array(labels)
repeat_seeds = list(range(config.SEED, config.SEED + N_REPEATS))
current_cnn_oof_by_repeat = [
    np.load(config.DATA_PROCESSED / f"rung3_oof_seed{s}.npy") for s in repeat_seeds
]

deltas = [
    evaluate.log_loss_score(y_true, oof_repeats_aug[i]) - evaluate.log_loss_score(y_true, current_cnn_oof_by_repeat[i])
    for i in range(N_REPEATS)
]
gate = evaluate.paired_repeat_gate(deltas)

print(f"per-repeat deltas (augmented - current CNN): {[f'{d:+.4f}' for d in deltas]}")
print(f"mean={gate['mean']:+.4f}, sd={gate['sd']:.4f}, "
      f"95% CI=[{gate['ci_low']:+.4f}, {gate['ci_high']:+.4f}]")
print(f"GATE {'PASSED' if gate['passed'] else 'NOT PASSED'}: "
      f"{'augmentation REPLACES the current CNN.' if gate['passed'] else 'does not beat the current CNN by more than noise -- keep the current CNN.'}")

**What we're looking for:** does flip + rotation + brightness-jitter
augmentation (in place of no augmentation in rungs 0-3), trained with a
longer patience budget (`AUGMENTED_PATIENCE=20` vs. `config.PATIENCE=10`
-- see the fold-0-sanity-check cell), beat the current validated CNN
(0.4520) by more than noise?

**What we found:** *(paste: the fold-0 sanity check number; the
5-repeat mean/sd; the `paired_repeat_gate` per-repeat deltas, mean, sd,
95% CI; the GATE PASSED/NOT PASSED line)*

**Decision / next step:** *(if the gate passed: this CNN replaces the
current one in the submission blend -- re-run the blend-weight
leave-one-repeat-out check against it, since w_cnn=0.70 was tuned for
the old CNN. If not: keep the current CNN, move on to experiment 4
[`class_weight="balanced"`, `notebooks/12_cnn_class_weight.ipynb`], and
log this as a negative result per the project's standing rule -- note
the mean/CI here, since a directionally positive but non-significant
result at 5 repeats may be worth re-running at higher N_REPEATS later,
see project memory project_dat_parkinson_rung4_gate_review.md's power
analysis.)*